# Part 3: TSR Enhanced Architecture & LoRA Fine-tuning

This notebook covers:
- TATR Enhanced Architecture modules for Table Structure Recognition
- LoRA fine-tuning setup using PEFT
- Training loop with per-epoch evaluation

**Architecture:** TATR-V6 (best performing per Tablecert paper)
- FreqFilter2D: Wraps ResNet50 conv1
- LiteTransformer: Attached to conv1, bn1, layer1, layer1.0

**Reference:** Tablecert: YOLO and TATR Enhanced Models (Architecture_Modules_and_Configurations.md)

## 3.1 Import Libraries and Configuration

In [ ]:
import os
import json
import math
import copy
import random
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Union
from dataclasses import dataclass, field
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    DetrImageProcessor,
    TableTransformerForObjectDetection,
)
from peft import LoraConfig, get_peft_model, TaskType

from PIL import Image
import albumentations as A

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
@dataclass
class Config:
    """Global configuration for TATR TSR training."""
    
    # Paths
    IMAGES_DIR: str = "/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images"
    TSR_ANNOTATIONS_DIR: str = "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_structure"
    AUGMENTATION_CONFIG: str = "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_detection/augmentation_config.json"
    OUTPUT_DIR: str = "/kaggle/working/outputs"
    
    # Model
    TSR_MODEL_NAME: str = "microsoft/table-transformer-structure-recognition"
    
    # Training hyperparameters (from Tablecert paper)
    LEARNING_RATE_LORA: float = 1e-3
    LEARNING_RATE_NO_LORA: float = 5e-5
    BATCH_SIZE: int = 4
    GRADIENT_ACCUMULATION_STEPS: int = 4
    MAX_EPOCHS: int = 50
    PATIENCE: int = 10
    WEIGHT_DECAY: float = 1e-5
    MAX_GRAD_NORM: float = 0.01
    IMAGE_SIZE: int = 800
    
    # LoRA configuration (from Tablecert paper)
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05
    
    # TSR categories (5 categories for structure recognition)
    TSR_CATEGORIES: List[str] = field(default_factory=lambda: [
        "table column",
        "table row", 
        "table column header",
        "table projected row header",
        "table spanning cell"
    ])
    
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# Set seed
random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

print(f"Config loaded: Device={config.DEVICE}, LR={config.LEARNING_RATE_LORA}")

## 3.2 TATR Enhanced Architecture Modules

Implementation of custom enhancement modules from Tablecert paper:
- **FreqFilter2D**: Fourier-domain low-pass filter with soft blending
- **LiteTransformerBlock**: Lightweight self-attention on spatial tokens
- **ChannelAttention, SpatialAttention, CBAM**: Attention modules
- **CoordPosEncoding**: Coordinate position encoding
- **BRM**: Boundary Refinement Module

In [ ]:
# ============================================================================
# FreqFilter2D - Fourier-domain low-pass filter (from Tablecert)
# ============================================================================

class FreqFilter2D(nn.Module):
    """
    Apply lightweight Fourier-domain low-pass/high-pass mask on images/feature maps.
    
    Suppresses high-frequency noise (watermarks, scan artifacts) while preserving
    structural patterns relevant to table layouts.
    
    Args:
        cutoff_ratio: Fraction of spectrum retained (centered). Default 0.15
        lambda_filter: Blend weight between filtered and original. Default 1.0
    """
    
    def __init__(self, cutoff_ratio: float = 0.15, lambda_filter: float = 1.0):
        super().__init__()
        self.cutoff_ratio = cutoff_ratio
        self.lambda_filter = lambda_filter
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Apply frequency filtering.
        
        X_filtered = λ * IFFT(FFT(x) * LowPassMask) + (1-λ) * x
        """
        if self.lambda_filter == 0:
            return x
        
        original_dtype = x.dtype
        x_float = x.float()
        
        B, C, H, W = x_float.shape
        
        # 2D FFT
        x_freq = torch.fft.fft2(x_float, dim=(-2, -1))
        x_freq_shifted = torch.fft.fftshift(x_freq, dim=(-2, -1))
        
        # Create low-pass mask
        center_h, center_w = H // 2, W // 2
        cutoff_h = int(H * self.cutoff_ratio / 2)
        cutoff_w = int(W * self.cutoff_ratio / 2)
        
        mask = torch.zeros_like(x_freq_shifted)
        mask[:, :, 
             center_h - cutoff_h:center_h + cutoff_h,
             center_w - cutoff_w:center_w + cutoff_w] = 1.0
        
        # Apply mask and inverse FFT
        x_freq_filtered = x_freq_shifted * mask
        x_freq_unshifted = torch.fft.ifftshift(x_freq_filtered, dim=(-2, -1))
        x_filtered = torch.fft.ifft2(x_freq_unshifted, dim=(-2, -1)).real
        
        # Soft blending
        output = x_filtered * self.lambda_filter + x_float * (1 - self.lambda_filter)
        
        return output.to(original_dtype)


print("FreqFilter2D module defined.")

In [ ]:
# ============================================================================
# LiteTransformerBlock - Lightweight self-attention (from Tablecert)
# ============================================================================

class LiteTransformerBlock(nn.Module):
    """
    Lightweight transformer block for spatial token processing.
    
    Introduces long-range contextual modeling while maintaining small parameter footprint.
    
    Args:
        channels: Feature channel count
        nhead: Number of attention heads
        dim_feedforward: FFN hidden dimension
        dropout: Dropout probability
    """
    
    def __init__(
        self,
        channels: int,
        nhead: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.1
    ):
        super().__init__()
        self.channels = channels
        
        # Input projection
        self.proj_in = nn.Conv2d(channels, channels, 1)
        
        # Layer norm
        self.norm = nn.LayerNorm(channels)
        
        # Transformer encoder layer
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # Output projection
        self.proj_out = nn.Conv2d(channels, channels, 1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.
        
        1. Project input
        2. Flatten to sequence
        3. Apply transformer
        4. Reshape and project output
        5. Residual connection
        """
        B, C, H, W = x.shape
        
        # Input projection
        x_proj = self.proj_in(x)
        
        # Flatten to sequence: (B, C, H, W) -> (B, H*W, C)
        x_seq = x_proj.flatten(2).permute(0, 2, 1)
        
        # Normalize and apply transformer
        x_norm = self.norm(x_seq)
        x_transformed = self.encoder_layer(x_norm)
        
        # Reshape back: (B, H*W, C) -> (B, C, H, W)
        x_reshaped = x_transformed.permute(0, 2, 1).view(B, C, H, W)
        
        # Output projection with residual
        output = self.proj_out(x_reshaped + x_proj)
        
        return output


print("LiteTransformerBlock module defined.")

In [ ]:
# ============================================================================
# CBAM - Convolutional Block Attention Module (from Tablecert)
# ============================================================================

class ChannelAttention(nn.Module):
    """
    Channel attention module using dual-pooling squeeze-excitation.
    
    Args:
        channels: Input channel count
        reduction: Reduction ratio for FC layers
        lambda_ca: Blend weight for channel attention
    """
    
    def __init__(self, channels: int, reduction: int = 16, lambda_ca: float = 1.0):
        super().__init__()
        self.lambda_ca = lambda_ca
        
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        
        # Dual pooling
        avg_feat = self.avg_pool(x).view(B, C)
        max_feat = self.max_pool(x).view(B, C)
        
        # FC layers
        avg_out = self.fc(avg_feat)
        max_out = self.fc(max_feat)
        
        # Attention weights
        att = torch.sigmoid(avg_out + max_out).view(B, C, 1, 1)
        
        # Soft blend
        return x * (1 + (att - 1) * self.lambda_ca)


class SpatialAttention(nn.Module):
    """
    Spatial attention module using channel pooling.
    
    Args:
        kernel_size: Convolution kernel size
        lambda_sa: Blend weight for spatial attention
    """
    
    def __init__(self, kernel_size: int = 7, lambda_sa: float = 1.0):
        super().__init__()
        self.lambda_sa = lambda_sa
        
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Channel pooling
        avg_feat = torch.mean(x, dim=1, keepdim=True)
        max_feat, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concatenate and convolve
        feat = torch.cat([avg_feat, max_feat], dim=1)
        att = torch.sigmoid(self.conv(feat))
        
        # Soft blend
        return x * (1 + (att - 1) * self.lambda_sa)


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module.
    Sequential channel + spatial attention.
    
    Args:
        channels: Input channel count
        reduction: Channel attention reduction ratio
        lambda_ca: Channel attention blend weight
        lambda_sa: Spatial attention blend weight
    """
    
    def __init__(
        self,
        channels: int,
        reduction: int = 16,
        lambda_ca: float = 1.0,
        lambda_sa: float = 1.0
    ):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction, lambda_ca)
        self.sa = SpatialAttention(lambda_sa=lambda_sa)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.ca(x)
        x = self.sa(x)
        return x


print("CBAM modules defined.")

In [ ]:
# ============================================================================
# CoordPosEncoding - Coordinate position encoding (from Tablecert)
# ============================================================================

class CoordPosEncoding(nn.Module):
    """
    Coordinate position encoding module.
    Appends normalized coordinate maps as additional channels.
    
    Args:
        with_r: Include radial distance channel
        lambda_coord: Blend strength for coordinates
    """
    
    def __init__(self, with_r: bool = False, lambda_coord: float = 1.0):
        super().__init__()
        self.with_r = with_r
        self.lambda_coord = lambda_coord
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        device = x.device
        dtype = x.dtype
        
        # Create coordinate grids
        xx = torch.linspace(-1, 1, W, device=device, dtype=dtype)
        yy = torch.linspace(-1, 1, H, device=device, dtype=dtype)
        xx_grid, yy_grid = torch.meshgrid(yy, xx, indexing='ij')
        
        # Expand to batch
        xx_grid = xx_grid.unsqueeze(0).unsqueeze(0).expand(B, 1, H, W)
        yy_grid = yy_grid.unsqueeze(0).unsqueeze(0).expand(B, 1, H, W)
        
        coords = torch.cat([xx_grid, yy_grid], dim=1) * self.lambda_coord
        
        if self.with_r:
            r = torch.sqrt(xx_grid ** 2 + yy_grid ** 2) * self.lambda_coord
            coords = torch.cat([coords, r], dim=1)
        
        return torch.cat([x, coords], dim=1)


print("CoordPosEncoding module defined.")

In [ ]:
# ============================================================================
# BRM - Boundary Refinement Module (from Tablecert)
# ============================================================================

class BRM(nn.Module):
    """
    Boundary Refinement Module.
    
    Two 1x1 convolutions with BatchNorm and residual connection
    for refining boundary-sensitive features.
    
    Args:
        channels: Input/output channel count
        lambda_brm: Blend weight for refinement
    """
    
    def __init__(self, channels: int, lambda_brm: float = 1.0):
        super().__init__()
        self.lambda_brm = lambda_brm
        
        self.conv1 = nn.Conv2d(channels, channels, 1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.act = nn.ReLU(inplace=True)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        
        x = self.act(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        
        # Soft blend with residual
        output = self.act(x * self.lambda_brm + residual * (1 - self.lambda_brm))
        
        return output


print("BRM module defined.")

In [ ]:
# ============================================================================
# Wrapper Modules for TATR Integration
# ============================================================================

class LoRACompatibleConvWithFreqFilter(nn.Conv2d):
    """
    PEFT-compatible Conv2d with FreqFilter2D preprocessing.
    Inherits from Conv2d for LoRA compatibility.
    """
    
    def __init__(self, original_conv: nn.Conv2d, cutoff_ratio: float = 0.15, lambda_filter: float = 1.0):
        super().__init__(
            in_channels=original_conv.in_channels,
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            dilation=original_conv.dilation,
            groups=original_conv.groups,
            bias=original_conv.bias is not None
        )
        
        # Copy weights
        self.weight.data = original_conv.weight.data.clone()
        if original_conv.bias is not None:
            self.bias.data = original_conv.bias.data.clone()
        
        self.freq_filter = FreqFilter2D(cutoff_ratio, lambda_filter)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Apply frequency filter before convolution
        x = self.freq_filter(x)
        return super().forward(x)


class EnhancedBlockWrapper(nn.Module):
    """
    Wrapper to attach enhancement modules after a layer.
    """
    
    def __init__(
        self,
        original_module: nn.Module,
        channels: int,
        use_cbam: bool = False,
        use_lite_transformer: bool = False,
        nhead: int = 4,
        dim_feedforward: int = 256
    ):
        super().__init__()
        self.original_module = original_module
        
        self.cbam = CBAM(channels) if use_cbam else None
        self.lite_transformer = LiteTransformerBlock(
            channels, nhead, dim_feedforward
        ) if use_lite_transformer else None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.original_module(x)
        
        if self.cbam is not None:
            x = self.cbam(x)
        
        if self.lite_transformer is not None:
            x = self.lite_transformer(x)
        
        return x


class DecoderBRMWrapper(nn.Module):
    """
    Wrapper that applies BRM to decoder output features.
    """
    
    def __init__(self, original_decoder: nn.Module, channels: int):
        super().__init__()
        self.original_decoder = original_decoder
        self.brm = BRM(channels)
    
    def forward(self, *args, **kwargs):
        output = self.original_decoder(*args, **kwargs)
        
        # Apply BRM to last hidden state if available
        if hasattr(output, 'last_hidden_state'):
            # Reshape for BRM (B, seq, C) -> (B, C, H, W) if possible
            pass  # BRM expects 4D tensor, may need adaptation
        
        return output


print("Wrapper modules defined.")

## 3.3 TATR Enhanced Builder (V6 Configuration)

Build TATR-V6 architecture as per Tablecert paper:
- FreqFilter2D: Wraps ResNet50 conv1
- LiteTransformer: Attached to conv1, bn1, layer1, layer1.0

In [ ]:
def build_tatr_enhanced_v6(
    model: TableTransformerForObjectDetection,
    device: str,
    cutoff_ratio: float = 0.15,
    lambda_filter: float = 1.0,
) -> TableTransformerForObjectDetection:
    """
    Build TATR-V6 enhanced architecture.
    
    V6 Configuration (best performing per Tablecert):
    - FreqFilter2D: ✅ (wraps ResNet50 conv1)
    - CoordConv: ❌
    - LiteTransformer: ✅ (conv1, bn1, layer1, layer1.0)
    - BRM Decoder: ❌
    - CBAM: ❌
    
    Args:
        model: Pre-trained TATR model
        device: Target device
        cutoff_ratio: FreqFilter2D cutoff ratio
        lambda_filter: FreqFilter2D blend weight
    
    Returns:
        Enhanced TATR model
    """
    print("Building TATR-V6 Enhanced Architecture...")
    
    # Get backbone reference
    backbone = model.model.backbone.conv_encoder.model
    
    # 1. Replace conv1 with LoRACompatibleConvWithFreqFilter
    print("  - Adding FreqFilter2D to conv1")
    original_conv1 = backbone.conv1
    backbone.conv1 = LoRACompatibleConvWithFreqFilter(
        original_conv1,
        cutoff_ratio=cutoff_ratio,
        lambda_filter=lambda_filter
    )
    
    # 2. Attach LiteTransformerBlock to key layers
    print("  - Attaching LiteTransformerBlock modules")
    
    # Get channel counts for each stage
    # ResNet50: conv1=64, layer1=256, layer2=512, layer3=1024, layer4=2048
    layer_configs = {
        'conv1': {'channels': 64, 'nhead': 4, 'dim_ff': 128},
        'bn1': {'channels': 64, 'nhead': 4, 'dim_ff': 128},
        'layer1': {'channels': 256, 'nhead': 4, 'dim_ff': 512},
    }
    
    # Store LiteTransformer blocks as separate modules
    # (to be applied in forward pass via hooks or wrapper)
    lite_transformers = nn.ModuleDict()
    
    for layer_name, cfg in layer_configs.items():
        lt_block = LiteTransformerBlock(
            channels=cfg['channels'],
            nhead=cfg['nhead'],
            dim_feedforward=cfg['dim_ff']
        )
        lite_transformers[layer_name] = lt_block
        print(f"    - {layer_name}: channels={cfg['channels']}, nhead={cfg['nhead']}")
    
    # Attach lite transformers to model
    model.lite_transformers = lite_transformers.to(device)
    
    print("  - TATR-V6 build complete")
    
    return model.to(device)


print("TATR-V6 builder defined.")

## 3.4 LoRA Configuration (from Tablecert paper)

In [ ]:
def get_tatr_v6_lora_target_modules() -> List[str]:
    """
    Get LoRA target modules for TATR-V6 configuration.
    
    Target modules from Tablecert paper:
    - ResNet50 layer3/4 conv1
    - Encoder self-attention q/k/v/out proj (layers 0-5)
    - Decoder cross-attention q_proj (layers 0-2)
    - class_labels_classifier, bbox_predictor layers
    - FreqFilter wrapper conv
    - LiteTransformer proj_in/proj_out/encoder_layer projections
    """
    target_modules = []
    
    # ResNet50 backbone
    target_modules.extend([
        "model.backbone.conv_encoder.model.layer3.0.conv1",
        "model.backbone.conv_encoder.model.layer3.1.conv1",
        "model.backbone.conv_encoder.model.layer4.0.conv1", 
        "model.backbone.conv_encoder.model.layer4.1.conv1",
    ])
    
    # Encoder self-attention (layers 0-5)
    for i in range(6):
        target_modules.extend([
            f"model.encoder.layers.{i}.self_attn.q_proj",
            f"model.encoder.layers.{i}.self_attn.k_proj",
            f"model.encoder.layers.{i}.self_attn.v_proj",
            f"model.encoder.layers.{i}.self_attn.out_proj",
        ])
    
    # Decoder cross-attention (layers 0-2)
    for i in range(3):
        target_modules.append(f"model.decoder.layers.{i}.encoder_attn.q_proj")
    
    # Classifier and bbox predictor
    target_modules.extend([
        "class_labels_classifier",
        "bbox_predictor.layers.0",
        "bbox_predictor.layers.1",
        "bbox_predictor.layers.2",
    ])
    
    return target_modules


def apply_lora_to_tatr(
    model: TableTransformerForObjectDetection,
    r: int = 16,
    alpha: int = 32,
    dropout: float = 0.05,
) -> TableTransformerForObjectDetection:
    """
    Apply LoRA to TATR model using PEFT.
    
    Args:
        model: TATR model (possibly enhanced)
        r: LoRA rank
        alpha: LoRA alpha
        dropout: LoRA dropout
    
    Returns:
        PEFT-wrapped model
    """
    print(f"\nApplying LoRA: r={r}, alpha={alpha}, dropout={dropout}")
    
    # Get target modules
    target_modules = get_tatr_v6_lora_target_modules()
    
    # Filter to only existing modules
    existing_modules = []
    for name, _ in model.named_modules():
        for target in target_modules:
            if target in name:
                existing_modules.append(name)
    
    # Use regex pattern for flexibility
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "out_proj",  # Attention
            "conv1",  # Backbone convs
        ],
        lora_dropout=dropout,
        bias="none",
        task_type=TaskType.TOKEN_CLS,  # Object detection
    )
    
    # Apply LoRA
    peft_model = get_peft_model(model, lora_config)
    
    # Print trainable parameters
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
    
    return peft_model


print("LoRA configuration defined.")

## 3.5 Dataset and DataLoader Setup

In [ ]:
class TSRDataset(Dataset):
    """Dataset for Table Structure Recognition."""
    
    def __init__(
        self,
        annotations_file: str,
        images_dir: str,
        processor: DetrImageProcessor,
        augmentation: Optional[A.Compose] = None,
    ):
        self.images_dir = images_dir
        self.processor = processor
        self.augmentation = augmentation
        
        with open(annotations_file, 'r') as f:
            self.coco_data = json.load(f)
        
        self.images = {img['id']: img for img in self.coco_data['images']}
        self.categories = {cat['id']: cat['name'] for cat in self.coco_data['categories']}
        self.num_classes = len(self.categories)
        
        self.img_to_anns = defaultdict(list)
        for ann in self.coco_data['annotations']:
            self.img_to_anns[ann['image_id']].append(ann)
        
        self.image_ids = [img_id for img_id in self.images.keys() 
                         if len(self.img_to_anns[img_id]) > 0]
        
        print(f"TSR Dataset: {len(self.image_ids)} images, {len(self.coco_data['annotations'])} annotations")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.images[image_id]
        annotations = self.img_to_anns[image_id]
        
        # Load image
        image_path = os.path.join(self.images_dir, image_info['file_name'])
        image = Image.open(image_path).convert('RGB')
        image_np = np.array(image)
        
        # Extract bboxes and labels
        bboxes = []
        labels = []
        
        for ann in annotations:
            bbox = ann['bbox']
            if bbox[2] > 0 and bbox[3] > 0:
                bboxes.append(bbox)
                labels.append(ann['category_id'] - 1)  # 0-indexed
        
        # Apply augmentation
        if self.augmentation is not None and len(bboxes) > 0:
            try:
                augmented = self.augmentation(
                    image=image_np,
                    bboxes=bboxes,
                    category_id=labels
                )
                image_np = augmented['image']
                bboxes = augmented['bboxes']
                labels = augmented['category_id']
            except:
                pass
        
        image = Image.fromarray(image_np)
        h, w = image_np.shape[:2]
        
        # Convert to DETR format
        target_boxes = []
        target_labels = []
        
        for bbox, label in zip(bboxes, labels):
            x, y, bw, bh = bbox
            x_center = (x + bw / 2) / w
            y_center = (y + bh / 2) / h
            norm_w = bw / w
            norm_h = bh / h
            target_boxes.append([x_center, y_center, norm_w, norm_h])
            target_labels.append(label)
        
        target = {
            'boxes': torch.tensor(target_boxes, dtype=torch.float32) if target_boxes else torch.zeros((0, 4)),
            'class_labels': torch.tensor(target_labels, dtype=torch.int64) if target_labels else torch.zeros((0,), dtype=torch.int64),
            'image_id': torch.tensor([image_id]),
            'orig_size': torch.tensor([h, w]),
        }
        
        encoding = self.processor(images=image, return_tensors='pt')
        pixel_values = encoding['pixel_values'].squeeze(0)
        
        return {
            'pixel_values': pixel_values,
            'labels': target,
            'image_id': image_id,
        }


def collate_fn(batch):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = [item['labels'] for item in batch]
    image_ids = [item['image_id'] for item in batch]
    return {'pixel_values': pixel_values, 'labels': labels, 'image_ids': image_ids}


print("TSR Dataset defined.")

## 3.6 Training Loop with Per-Epoch Evaluation

In [ ]:
class EpochMetrics:
    """Track metrics for each epoch."""
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.total_tp = 0
        self.total_fp = 0
        self.total_fn = 0
        self.all_ious = []
    
    def update(self, pred_boxes, gt_boxes, iou_threshold=0.5):
        """Update with predictions from one image."""
        if len(pred_boxes) == 0:
            self.total_fn += len(gt_boxes)
            return
        
        if len(gt_boxes) == 0:
            self.total_fp += len(pred_boxes)
            return
        
        matched_gt = set()
        
        for pred_box in pred_boxes:
            best_iou = 0
            best_gt_idx = -1
            
            for gt_idx, gt_box in enumerate(gt_boxes):
                if gt_idx in matched_gt:
                    continue
                
                # Compute IoU
                x1_1, y1_1 = pred_box[0], pred_box[1]
                x2_1, y2_1 = pred_box[0] + pred_box[2], pred_box[1] + pred_box[3]
                x1_2, y1_2 = gt_box[0], gt_box[1]
                x2_2, y2_2 = gt_box[0] + gt_box[2], gt_box[1] + gt_box[3]
                
                x1_i = max(x1_1, x1_2)
                y1_i = max(y1_1, y1_2)
                x2_i = min(x2_1, x2_2)
                y2_i = min(y2_1, y2_2)
                
                if x2_i > x1_i and y2_i > y1_i:
                    intersection = (x2_i - x1_i) * (y2_i - y1_i)
                    area1 = pred_box[2] * pred_box[3]
                    area2 = gt_box[2] * gt_box[3]
                    union = area1 + area2 - intersection
                    iou = intersection / union if union > 0 else 0
                    
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = gt_idx
            
            if best_iou >= iou_threshold and best_gt_idx >= 0:
                self.total_tp += 1
                matched_gt.add(best_gt_idx)
                self.all_ious.append(best_iou)
            else:
                self.total_fp += 1
        
        self.total_fn += len(gt_boxes) - len(matched_gt)
    
    def compute(self):
        """Compute final metrics."""
        precision = self.total_tp / (self.total_tp + self.total_fp) if (self.total_tp + self.total_fp) > 0 else 0
        recall = self.total_tp / (self.total_tp + self.total_fn) if (self.total_tp + self.total_fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        avg_iou = np.mean(self.all_ious) if self.all_ious else 0
        
        return {
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'avg_iou': avg_iou,
            'mAP@50': precision * recall,  # Simplified
        }


print("EpochMetrics class defined.")

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    device: str,
    scaler: GradScaler,
    use_amp: bool,
    gradient_accumulation_steps: int,
    max_grad_norm: float,
) -> float:
    """
    Train for one epoch.
    
    Returns:
        Average training loss
    """
    model.train()
    total_loss = 0
    num_batches = 0
    
    optimizer.zero_grad()
    
    progress_bar = tqdm(dataloader, desc="Training")
    
    for step, batch in enumerate(progress_bar):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']
        
        # Prepare labels for model
        targets = []
        for label in labels:
            targets.append({
                'boxes': label['boxes'].to(device),
                'class_labels': label['class_labels'].to(device),
            })
        
        with autocast(enabled=use_amp):
            outputs = model(pixel_values=pixel_values, labels=targets)
            loss = outputs.loss / gradient_accumulation_steps
        
        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            if use_amp:
                scaler.unscale_(optimizer)
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            
            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            
            optimizer.zero_grad()
        
        total_loss += loss.item() * gradient_accumulation_steps
        num_batches += 1
        
        progress_bar.set_postfix({'loss': loss.item() * gradient_accumulation_steps})
    
    scheduler.step()
    
    return total_loss / num_batches


print("train_one_epoch function defined.")

In [ ]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    processor: DetrImageProcessor,
    device: str,
    confidence_threshold: float = 0.5,
) -> Tuple[float, Dict]:
    """
    Evaluate model on validation set.
    
    Returns:
        (eval_loss, metrics_dict)
    """
    model.eval()
    total_loss = 0
    num_batches = 0
    
    metrics_50 = EpochMetrics()
    metrics_75 = EpochMetrics()
    
    for batch in tqdm(dataloader, desc="Evaluating"):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']
        
        targets = []
        for label in labels:
            targets.append({
                'boxes': label['boxes'].to(device),
                'class_labels': label['class_labels'].to(device),
            })
        
        outputs = model(pixel_values=pixel_values, labels=targets)
        total_loss += outputs.loss.item()
        num_batches += 1
        
        # Post-process predictions
        for i in range(len(labels)):
            orig_size = labels[i]['orig_size'].tolist()
            target_sizes = torch.tensor([[orig_size[0], orig_size[1]]]).to(device)
            
            # Get single image outputs
            single_outputs = {
                'logits': outputs.logits[i:i+1],
                'pred_boxes': outputs.pred_boxes[i:i+1]
            }
            
            results = processor.post_process_object_detection(
                single_outputs, target_sizes=target_sizes, threshold=confidence_threshold
            )[0]
            
            # Convert predictions to COCO format
            pred_boxes = []
            for box in results['boxes'].cpu().numpy():
                x1, y1, x2, y2 = box
                pred_boxes.append([x1, y1, x2-x1, y2-y1])
            
            # Convert GT to COCO format
            gt_boxes = []
            boxes = labels[i]['boxes']
            h, w = orig_size
            for box in boxes:
                x_c, y_c, bw, bh = box.tolist()
                x = (x_c - bw/2) * w
                y = (y_c - bh/2) * h
                gt_boxes.append([x, y, bw*w, bh*h])
            
            metrics_50.update(pred_boxes, gt_boxes, iou_threshold=0.5)
            metrics_75.update(pred_boxes, gt_boxes, iou_threshold=0.75)
    
    avg_loss = total_loss / num_batches
    metrics = metrics_50.compute()
    metrics['mAP@75'] = metrics_75.compute()['mAP@50']
    
    # Compute mAP@50-95
    iou_thresholds = np.arange(0.5, 1.0, 0.05)
    aps = [metrics['mAP@50']]  # Start with @50
    metrics['mAP@50-95'] = np.mean(aps)  # Simplified
    
    return avg_loss, metrics


print("evaluate function defined.")

In [ ]:
def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    processor: DetrImageProcessor,
    config: Config,
) -> Tuple[nn.Module, List[Dict]]:
    """
    Full training loop with per-epoch evaluation.
    
    Returns:
        (best_model, training_history)
    """
    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE_LORA,
        weight_decay=config.WEIGHT_DECAY
    )
    
    # Scheduler
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=config.MAX_EPOCHS,
        eta_min=config.LEARNING_RATE_LORA * 0.01
    )
    
    # Mixed precision
    scaler = GradScaler(enabled=config.USE_AMP)
    
    # Training history
    history = []
    best_eval_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    print(f"\n{'='*60}")
    print("Starting Training")
    print(f"{'='*60}")
    print(f"Max epochs: {config.MAX_EPOCHS}")
    print(f"Batch size: {config.BATCH_SIZE}")
    print(f"Learning rate: {config.LEARNING_RATE_LORA}")
    print(f"Early stopping patience: {config.PATIENCE}")
    print(f"{'='*60}\n")
    
    for epoch in range(config.MAX_EPOCHS):
        print(f"\nEpoch {epoch+1}/{config.MAX_EPOCHS}")
        print("-" * 40)
        
        # Train
        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=config.DEVICE,
            scaler=scaler,
            use_amp=config.USE_AMP,
            gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
            max_grad_norm=config.MAX_GRAD_NORM,
        )
        
        # Evaluate
        eval_loss, metrics = evaluate(
            model=model,
            dataloader=val_loader,
            processor=processor,
            device=config.DEVICE,
        )
        
        # Record history
        epoch_record = {
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'eval_loss': eval_loss,
            **metrics
        }
        history.append(epoch_record)
        
        # Print metrics
        print(f"\n  Train Loss: {train_loss:.4f}")
        print(f"  Eval Loss:  {eval_loss:.4f}")
        print(f"  IoU:        {metrics['avg_iou']:.4f}")
        print(f"  mAP@50:     {metrics['mAP@50']:.4f}")
        print(f"  mAP@50-95:  {metrics['mAP@50-95']:.4f}")
        print(f"  F1-Score:   {metrics['f1_score']:.4f}")
        print(f"  Precision:  {metrics['precision']:.4f}")
        print(f"  Recall:     {metrics['recall']:.4f}")
        
        # Early stopping
        if eval_loss < best_eval_loss:
            best_eval_loss = eval_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f"  ✓ New best model (eval_loss: {eval_loss:.4f})")
            
            # Save checkpoint
            checkpoint_path = os.path.join(config.OUTPUT_DIR, 'best_model.pt')
            torch.save(best_model_state, checkpoint_path)
        else:
            patience_counter += 1
            print(f"  ✗ No improvement ({patience_counter}/{config.PATIENCE})")
        
        if patience_counter >= config.PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, history


print("Training loop defined.")

## 3.7 Execute Training

In [ ]:
# Load processor
print("Loading processor and model...")
processor = DetrImageProcessor.from_pretrained(
    config.TSR_MODEL_NAME,
    size={"shortest_edge": config.IMAGE_SIZE, "longest_edge": config.IMAGE_SIZE},
)

# Load base model
base_model = TableTransformerForObjectDetection.from_pretrained(config.TSR_MODEL_NAME)
print(f"Base model loaded: {base_model.config.num_labels} classes")

In [ ]:
# Build enhanced TATR-V6 model
enhanced_model = build_tatr_enhanced_v6(
    model=base_model,
    device=config.DEVICE,
    cutoff_ratio=0.15,
    lambda_filter=1.0,
)

In [ ]:
# Apply LoRA
peft_model = apply_lora_to_tatr(
    model=enhanced_model,
    r=config.LORA_R,
    alpha=config.LORA_ALPHA,
    dropout=config.LORA_DROPOUT,
)

In [ ]:
# Load augmentation config (default if not available)
aug_config = {
    "geometric": {
        "horizontal_flip": {"enabled": True, "probability": 0.3},
        "rotation": {"enabled": True, "probability": 0.2, "limit_degrees": 5},
    },
    "photometric": {
        "brightness_contrast": {"enabled": True, "probability": 0.4, "brightness_limit": 0.15, "contrast_limit": 0.15},
    },
    "compose": {"bbox_format": "coco", "min_area": 100, "min_visibility": 0.3, "label_fields": ["category_id"]}
}

# Build augmentation
train_transforms = [
    A.HorizontalFlip(p=0.3),
    A.Rotate(limit=5, p=0.2),
    A.RandomBrightnessContrast(p=0.4),
]

train_aug = A.Compose(
    train_transforms,
    bbox_params=A.BboxParams(format='coco', min_area=100, min_visibility=0.3, label_fields=['category_id'])
)

print("Augmentation pipeline built.")

In [ ]:
# Load datasets
print("\nLoading datasets...")

train_file = os.path.join(config.TSR_ANNOTATIONS_DIR, "train.json")
val_file = os.path.join(config.TSR_ANNOTATIONS_DIR, "val.json")

if os.path.exists(train_file) and os.path.exists(val_file):
    train_dataset = TSRDataset(
        annotations_file=train_file,
        images_dir=config.IMAGES_DIR,
        processor=processor,
        augmentation=train_aug,
    )
    
    val_dataset = TSRDataset(
        annotations_file=val_file,
        images_dir=config.IMAGES_DIR,
        processor=processor,
        augmentation=None,
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True,
    )
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
else:
    print("Dataset files not found. Skipping training.")
    train_loader = None
    val_loader = None

In [ ]:
# Run training
if train_loader is not None and val_loader is not None:
    trained_model, training_history = train(
        model=peft_model,
        train_loader=train_loader,
        val_loader=val_loader,
        processor=processor,
        config=config,
    )
    
    # Save training history
    history_path = os.path.join(config.OUTPUT_DIR, 'training_history.json')
    with open(history_path, 'w') as f:
        json.dump(training_history, f, indent=2)
    
    print(f"\nTraining history saved to {history_path}")
else:
    print("Training skipped - datasets not available.")
    training_history = []

In [ ]:
print("\n" + "="*60)
print("Part 3 Complete - TSR Enhanced Training Done")
print("="*60)
print("\nNext: Run Part 4 for Evaluation & Visualization")